In [1]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/ECGPipes/notebooks
/Users/peli/Projects/Repositories/ECGPipes
Working Dir Base: /Users/peli/Projects/Repositories/ECGPipes


In [2]:
from nipype import Workflow, Node, MapNode, Function, IdentityInterface
from nipype.interfaces.io import SelectFiles, DataSink
# import
from src.preprocessing import *

loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


In [3]:
wf = Workflow(name="megpreproc")
# Create the nodes
# create the subject_id "injector"?
basedir = "data/ds006629"
wf.base_dir = "/scratch/workdir"
subject_list = ['sub-01','sub-02']
infosource = Node(IdentityInterface(fields=['subject_id']),
                  name="infosource")
infosource.iterables = [('subject_id', subject_list)]

templates = {"meg": "{subject_id}/meg/{subject_id}_task-MMNHCS_run-0_meg.fif"}
# create the fileselector
selectraw = Node(
    SelectFiles(templates, base_directory=basedir),
    name="selectfiles"
)

# create the processing nodes
crop = Node(Function(
    input_names=["in_file", "stim_channel", "min_buffer", "max_buffer"],
    output_names=["out_file"],
    function=crop_data
), name='CropData')

filter_node = Node(Function(
    input_names=["in_file", "l_freq", "h_freq"],
    output_names=["out_file"],
    function=filter_data
), name="FilterData")

grad_comp = Node(Function(
    input_names=["in_file", "auto", "order"],
    output_names=["out_file"],
    function=gradient_compensation
), name="GradientComp")

# IO
# Datasink - creates output folder for important outputs
datasink = Node(DataSink(base_directory="/output",
                         container="datasink"),
                name="datasink")

wf.connect([

    (infosource, selectraw, [("subject_id", "subject_id")]),

    (selectraw, crop, [("meg", "in_file")]),

    (crop, filter_node, [("out_file", "in_file")]),

    (filter_node, grad_comp, [("out_file", "in_file")]),

    (grad_comp, datasink, [("out_file", "megpreproc.@final")]),

])